<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/IMG2VID-WAN22-A14B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image to Video — Wan 2.2 A14B (best quality, L4 / A100)

Turns a still image into a short video with **Wan 2.2 Image-to-Video A14B**, the best open image-to-video model
for Colab. Needs an **L4 or A100** GPU (compute units). On the free T4 use
[IMG2VID-WAN22-5B-FREE.ipynb](https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/IMG2VID-WAN22-5B-FREE.ipynb).

1. **Runtime → Change runtime type → L4 GPU** (or A100).
2. Run **cell 1** (install + downloads, ~25 GB on an L4 — several minutes), **cell 2** (saves the server program) and
   **cell 3** (starts it and prints the `READY:` URL).
3. On the website open the **Image to Video** tab and paste the URL.

*Fast* mode uses the 4-step Lightning LoRAs (a few minutes per 5-second clip on an L4); *Quality* mode uses 20 steps
(much slower). Cell 1 picks Q4_K_M on an L4 and Q8_0 on an A100. Cell 4 shows the server log and memory use.
Don't run another GPU notebook at the same time.

**Sound:** cell 1 also installs **MMAudio**, which can add matching sound effects and ambience (footsteps, waves, wind, engines…) to each video — turn on *Add sound* on the website. It takes about another minute per clip. MMAudio is for non-commercial use (CC BY-NC 4.0); set `ADD_SOUND = False` in cell 1 to skip it.

In [ ]:
# Cell 1: install ComfyUI + ComfyUI-GGUF, check the GPU and download the image-to-video model
FAMILY = "a14b"
%cd /content
import os, json, subprocess
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI || true
!cd /content/ComfyUI && git pull -q || true
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q gguf fastapi uvicorn requests psutil
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-GGUF || git clone -q https://github.com/city96/ComfyUI-GGUF.git)
!cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF && pip install -q -r requirements.txt
!apt install -qqy aria2 ffmpeg > /dev/null

# ---- GPU
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().split("\n")[0]
except FileNotFoundError:  # nvidia-smi only exists when a GPU is attached
    gpu = ""
if "," not in gpu:
    raise RuntimeError(
        "This Colab runtime has no GPU, and video generation needs one.\n"
        "Fix: Runtime → Change runtime type → a GPU → Save, then run the cells again.\n"
        "If Colab can't connect to a GPU, your GPU time or compute units have run out for now (check Resources).")
gpu_name, vram_mib = [x.strip() for x in gpu.rsplit(",", 1)]
GPU = {"name": gpu_name, "vram_gib": round(int(vram_mib) / 1024, 1)}

# ---- Model: Wan 2.2 Image-to-Video A14B (two experts: high-noise for motion, low-noise for detail)
#      GGUF from QuantStack, 4-step Lightning LoRAs + VAE from Comfy-Org, text encoder from city96
if GPU["vram_gib"] < 20:
    raise RuntimeError(
        f"This GPU ({GPU['name']}, {GPU['vram_gib']} GB) is too small for Wan 2.2 A14B — it needs an L4 or A100.\n"
        "Pick Runtime → Change runtime type → L4 GPU (needs compute units), or on the free T4 use "
        "IMG2VID-WAN22-5B-FREE.ipynb: https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/IMG2VID-WAN22-5B-FREE.ipynb")
QUANT = "Q8_0" if GPU["vram_gib"] >= 38 else "Q4_K_M"   # A100: best quality; L4: fits with room for 81+ frames
HF_A14B = "https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main"
HF_WAN = "https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files"
FILES = {
    "high": (f"Wan2.2-I2V-A14B-HighNoise-{QUANT}.gguf", f"{HF_A14B}/HighNoise/Wan2.2-I2V-A14B-HighNoise-{QUANT}.gguf", "unet"),
    "low": (f"Wan2.2-I2V-A14B-LowNoise-{QUANT}.gguf", f"{HF_A14B}/LowNoise/Wan2.2-I2V-A14B-LowNoise-{QUANT}.gguf", "unet"),
    "lora_high": ("wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", f"{HF_WAN}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "loras"),
    "lora_low": ("wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", f"{HF_WAN}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "loras"),
    "text_encoder": ("umt5-xxl-encoder-Q5_K_M.gguf", "https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q5_K_M.gguf", "text_encoders"),
    "vae": ("wan_2.1_vae.safetensors", f"{HF_WAN}/vae/wan_2.1_vae.safetensors", "vae"),
}
DEFAULTS = {
    "fps": 16, "max_frames": 121, "seconds": 5, "size": "720p" if GPU["vram_gib"] >= 38 else "480p",
    "fast": {"steps": 4, "cfg": 1.0, "shift": 5.0, "sampler": "euler", "scheduler": "simple"},      # Lightning LoRAs
    "quality": {"steps": 20, "cfg": 3.5, "shift": 8.0, "sampler": "euler", "scheduler": "simple"},  # plain model
}
LABEL = f"Wan 2.2 I2V A14B ({QUANT})"

# ---- Sound (optional): MMAudio adds matching sound effects / ambience to finished videos.
#      Set ADD_SOUND = False to skip it (saves ~8 GB of downloads). Model licence: CC BY-NC 4.0 (non-commercial).
ADD_SOUND = True
SOUND_OK = False
if ADD_SOUND:
    print("Installing MMAudio (sound for videos)...")
    !test -d /content/MMAudio || git clone -q https://github.com/hkchengrex/MMAudio /content/MMAudio
    !cd /content/MMAudio && git fetch -q origin && git checkout -q 974010a
    !pip install -q torchdiffeq open_clip_torch timm av einops librosa omegaconf colorlog
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("apple/DFN5B-CLIP-ViT-H-14-384", allow_patterns=["open_clip*"])
        snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", allow_patterns=["config.json", "bigvgan_generator.pt"])
        subprocess.run(["python", "-c", "from mmaudio.eval_utils import all_model_cfg as c; c['large_44k_v2'].download_if_needed()"],
                       cwd="/content/MMAudio", check=True)
        SOUND_OK = True
    except Exception as e:
        print(f"Sound is turned off: MMAudio could not be set up ({e}). Videos still work, just silent.")

# ---- Download (resumes if interrupted) and write the config the server reads
for key, (name, url, folder) in FILES.items():
    print(f"Downloading {name}...")
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url,
                    "-d", f"/content/ComfyUI/models/{folder}", "-o", name], check=True)
json.dump({"family": FAMILY, "label": LABEL, "gpu": GPU, "defaults": DEFAULTS, "sound": SOUND_OK,
           "files": {k: v[0] for k, v in FILES.items()}}, open("/content/video_config.json", "w"), indent=1)
print(f"GPU: {GPU['name']} ({GPU['vram_gib']} GB) -> {LABEL}, sound {'on' if SOUND_OK else 'off'}. Now run cells 2 and 3.")

In [ ]:
%%writefile /content/video_server.py
# Cell 2: the image-to-video server, saved as its own program (/content/video_server.py).
# Cell 3 runs it separately from this notebook: if a video runs out of memory only this program dies,
# and cell 3 restarts it automatically (same URL) instead of the whole Colab session crashing.
# Running this cell only saves the file; edit it here and re-run cells 2 and 3 to apply changes.
import os
import sys
import json

os.chdir("/content/ComfyUI")
sys.path.insert(0, "/content/ComfyUI")

CRASH_FILE = "/content/server_crash.txt"
CONFIG = json.load(open("/content/video_config.json"))  # written by cell 1: model files, GPU, defaults
FAMILY = CONFIG["family"]  # "a14b" (Wan 2.2 I2V A14B, best quality) or "5b" (Wan 2.2 TI2V 5B, free T4)


# ---------------------------------------------------------------- sound (MMAudio)
# The server runs this same file as a short-lived separate program to add sound to a finished video:
#   python video_server.py --add-sound in.mp4 out.mp4 "sound prompt" seed
# Running it separately means all of MMAudio's memory is released afterwards.
def add_sound_program(src, dst, prompt, seed):
    import subprocess
    os.chdir("/content/MMAudio")  # its weights live in ./weights and ./ext_weights
    sys.path[:] = ["/content/MMAudio"] + [p for p in sys.path if "ComfyUI" not in p]
    import numpy as np
    import torch
    from mmaudio.eval_utils import all_model_cfg, generate, load_video
    from mmaudio.model.flow_matching import FlowMatching
    from mmaudio.model.networks import get_my_mmaudio
    from mmaudio.model.utils.features_utils import FeaturesUtils

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    cfg = all_model_cfg["large_44k_v2"]
    cfg.download_if_needed()
    seq = cfg.seq_cfg
    # bf16 is fast on L4/A100; the T4 has no bf16, so it runs in full precision there
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32
    with torch.inference_mode():
        net = get_my_mmaudio(cfg.model_name).to("cuda", dtype).eval()
        net.load_weights(torch.load(cfg.model_path, map_location="cuda", weights_only=True))
        feats = FeaturesUtils(tod_vae_ckpt=cfg.vae_path, synchformer_ckpt=cfg.synchformer_ckpt,
                              enable_conditions=True, mode=cfg.mode, bigvgan_vocoder_ckpt=cfg.bigvgan_16k_path,
                              need_vae_encoder=False).to("cuda", dtype).eval()
        info = load_video(src, 30.0, load_all_frames=False)  # the whole clip (videos here are shorter than 30 s)
        seq.duration = info.duration_sec
        net.update_seq_lengths(seq.latent_seq_len, seq.clip_seq_len, seq.sync_seq_len)
        rng = torch.Generator(device="cuda")
        rng.manual_seed(seed)
        fm = FlowMatching(min_sigma=0, inference_mode="euler", num_steps=25)
        audio = generate(info.clip_frames.unsqueeze(0), info.sync_frames.unsqueeze(0), [prompt],
                         negative_text=["music, speech, talking, narration, distortion, noise"],
                         feature_utils=feats, net=net, fm=fm, rng=rng, cfg_strength=4.5)
    wave = audio.float().cpu()[0].numpy()
    if wave.ndim == 1:
        wave = wave[None]
    pcm = np.ascontiguousarray(wave.T.clip(-1, 1).astype("<f4"))
    # keep the video stream as it is and add the sound as AAC (plays in every browser)
    proc = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-f", "f32le",
                             "-ar", str(seq.sampling_rate), "-ac", str(wave.shape[0]), "-i", "-",
                             "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                             "-shortest", "-movflags", "+faststart", dst], stdin=subprocess.PIPE)
    proc.stdin.write(pcm.tobytes())
    proc.stdin.close()
    if proc.wait() != 0:
        raise RuntimeError("ffmpeg could not add the sound")
    print(f"Sound added: {info.duration_sec:.1f} s, peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GB", flush=True)


if len(sys.argv) > 1 and sys.argv[1] == "--add-sound":
    add_sound_program(sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5]))
    sys.exit(0)


def last_crash_message():
    """Set by cell 3 when the server crashed and was restarted, so the website can say why a job vanished."""
    try:
        return open(CRASH_FILE).read().strip()
    except OSError:
        return ""


import gc
import io
import re
import time
import uuid
import base64
import random
import logging
import warnings
import threading
import traceback
import subprocess
import importlib.util

import numpy as np
import requests
import torch
from PIL import Image, ImageOps

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import nodes
import comfy.utils
import comfy.model_management
from comfy_extras import nodes_wan, nodes_model_advanced

# ComfyUI-GGUF (custom nodes are not auto-loaded when ComfyUI is used as a library)
if "UnetLoaderGGUF" not in nodes.NODE_CLASS_MAPPINGS:
    gguf_path = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/__init__.py"
    spec = importlib.util.spec_from_file_location("ComfyUI-GGUF", gguf_path,
                                                  submodule_search_locations=[os.path.dirname(gguf_path)])
    module = importlib.util.module_from_spec(spec)
    sys.modules["ComfyUI-GGUF"] = module
    spec.loader.exec_module(module)
    nodes.NODE_CLASS_MAPPINGS.update(getattr(module, "NODE_CLASS_MAPPINGS", {}))

M = nodes.NODE_CLASS_MAPPINGS
UnetLoaderGGUF, CLIPLoaderGGUF = M["UnetLoaderGGUF"](), M["CLIPLoaderGGUF"]()
VAELoader, LoraLoaderModelOnly = M["VAELoader"](), M["LoraLoaderModelOnly"]()
CLIPTextEncode, VAEDecode = M["CLIPTextEncode"](), M["VAEDecode"]()
KSampler, KSamplerAdvanced = M["KSampler"](), M["KSamplerAdvanced"]()
ModelSamplingSD3 = nodes_model_advanced.ModelSamplingSD3()
F = CONFIG["files"]
D = CONFIG["defaults"]

# Two modes: "fast" uses the 4-step LoRA(s), "quality" the plain model with more steps
print(f"Loading {CONFIG['label']}...", flush=True)
with torch.inference_mode():
    clip = CLIPLoaderGGUF.load_clip(F["text_encoder"], "wan")[0]
    vae = VAELoader.load_vae(F["vae"])[0]
    if FAMILY == "a14b":
        base_high = UnetLoaderGGUF.load_unet(F["high"])[0]
        base_low = UnetLoaderGGUF.load_unet(F["low"])[0]
        fast_high = LoraLoaderModelOnly.load_lora_model_only(base_high, F["lora_high"], 1.0)[0]
        fast_low = LoraLoaderModelOnly.load_lora_model_only(base_low, F["lora_low"], 1.0)[0]
        MODELS = {
            "fast": tuple(ModelSamplingSD3.patch(m, D["fast"]["shift"])[0] for m in (fast_high, fast_low)),
            "quality": tuple(ModelSamplingSD3.patch(m, D["quality"]["shift"])[0] for m in (base_high, base_low)),
        }
    else:
        base = UnetLoaderGGUF.load_unet(F["model"])[0]
        fast = LoraLoaderModelOnly.load_lora_model_only(base, F["lora_turbo"], 1.0)[0]
        MODELS = {
            "fast": ModelSamplingSD3.patch(fast, D["fast"]["shift"])[0],
            "quality": ModelSamplingSD3.patch(base, D["quality"]["shift"])[0],
        }
print("Models loaded.", flush=True)

DEFAULT_NEGATIVE = ("blurry, low quality, worst quality, jpeg artifacts, static, still image, frozen, "
                    "deformed, distorted face, extra fingers, bad hands, watermark, text, subtitles")
SIZE_AREAS = {"480p": 832 * 480, "720p": 1280 * 720}
MULTIPLE = 16 if FAMILY == "a14b" else 32  # the 5B model's latents are 16x smaller, so sizes must be /32
END_FRAME_OK = FAMILY == "a14b"  # first + last frame: Wan 2.2 A14B I2V only (the 5B model takes a start image only)
VIDEO_DIR = "/content/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)


def video_size(img_w, img_h, size):
    """Keep the start image's shape at about the chosen resolution's pixel count."""
    area = SIZE_AREAS.get(size, SIZE_AREAS["480p"])
    scale = (area / (img_w * img_h)) ** 0.5
    w = max(MULTIPLE * 8, int(round(img_w * scale / MULTIPLE)) * MULTIPLE)
    h = max(MULTIPLE * 8, int(round(img_h * scale / MULTIPLE)) * MULTIPLE)
    return w, h


def frame_count(seconds, fps):
    """Wan needs 4n+1 frames."""
    n = int(round(float(seconds) * fps))
    n = max(17, min(n, D["max_frames"]))
    return (n - 1) // 4 * 4 + 1


def load_image(value):
    if not value:
        raise ValueError("Add a start image first")
    value = value.strip()
    if re.match(r"^https?://", value, re.I):
        r = requests.get(value, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        raw = r.content
    else:
        raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", value))
    return ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")


def fit_to(pil, w, h):
    """Scale and centre-crop a picture to exactly w x h (used for the end frame)."""
    return ImageOps.fit(pil, (w, h), Image.Resampling.LANCZOS, centering=(0.5, 0.5))


def frame_data_url(frame):
    """One decoded frame [H, W, 3] 0..1 -> PNG data URL (the exact last frame, for "Continue from last frame")."""
    arr = (frame.cpu().numpy().clip(0, 1) * 255).round().astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def save_mp4(frames, fps, path):
    """frames: [T, H, W, 3] float 0..1 -> H.264 MP4 (plays in every browser)."""
    arr = (frames.cpu().numpy().clip(0, 1) * 255).round().astype(np.uint8)
    t, h, w, _ = arr.shape
    proc = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
                             "-s", f"{w}x{h}", "-r", str(fps), "-i", "-", "-c:v", "libx264",
                             "-pix_fmt", "yuv420p", "-crf", "18", "-movflags", "+faststart", path],
                            stdin=subprocess.PIPE)
    proc.stdin.write(arr.tobytes())
    proc.stdin.close()
    if proc.wait() != 0:
        raise RuntimeError("ffmpeg could not write the video")
    return t


SOUND_OK = bool(CONFIG.get("sound"))  # cell 1 installed MMAudio


def add_sound(job, path, prompt, seed):
    """Runs MMAudio on the finished video (in its own program). Returns None, or why it failed."""
    job["stage"] = "adding sound"
    job["step"], job["total_steps"] = 0, 0
    comfy.model_management.unload_all_models()  # free the GPU for MMAudio; the video model reloads next time
    comfy.model_management.soft_empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    out = path[:-4] + "_sound.mp4"
    try:
        r = subprocess.run([sys.executable, os.path.abspath(__file__), "--add-sound", path, out, prompt,
                            str(seed % 2**31)], capture_output=True, text=True, timeout=1200)
    except subprocess.TimeoutExpired:
        return "adding sound took too long"
    if r.returncode != 0 or not os.path.exists(out):
        print(r.stdout[-2000:], r.stderr[-4000:], flush=True)
        if r.returncode < 0 or "Killed" in r.stderr or "out of memory" in r.stderr.lower():
            return "ran out of memory while adding sound"
        last = [l for l in r.stderr.strip().splitlines() if l.strip()][-1:] or ["unknown error"]
        return last[0][:200]
    os.replace(out, path)
    print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "Sound added.", flush=True)
    return None


# ---------------------------------------------------------------- jobs
JOBS = {}
gpu_lock = threading.Lock()
current_job = {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def new_job(**extra):
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 3600]:
        JOBS.pop(jid, None)
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"status": "queued", "created": now, "error": None, "stage": "queued", "step": 0,
                    "total_steps": 0, **extra}
    return job_id


def memory_report():
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = f"RAM {(vm.total - vm.available) / 1e9:.1f}/{vm.total / 1e9:.1f} GB"
    except Exception:
        ram = "RAM ?"
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        return f"{ram}, VRAM {(total - free) / 1e9:.1f}/{total / 1e9:.1f} GB"
    return ram


def run_video(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        current_job["id"] = job_id
        job["status"] = "running"
        t0 = time.time()
        try:
            mode = "quality" if req.mode == "quality" else "fast"
            P = D[mode]
            fps = D["fps"]
            pil = load_image(req.image)
            w, h = video_size(pil.width, pil.height, req.size)
            length = frame_count(req.seconds, fps)
            steps = max(1, min(int(req.steps or P["steps"]), 50))
            cfg = float(P["cfg"] if req.cfg is None else req.cfg)
            seed = req.seed if req.seed and req.seed > 0 else random.randint(1, 2**63 - 1)
            job.update(width=w, height=h, frames=length, fps=fps, seed=seed, mode=mode)
            print(f"[{job_id}] {mode}: {w}x{h}, {length} frames @ {fps} fps, {steps} steps, cfg {cfg}"
                  + (", with end frame" if req.end_image and END_FRAME_OK else ""), flush=True)
            start = torch.from_numpy(np.array(pil.resize((w, h), Image.Resampling.LANCZOS)).astype(np.float32) / 255.0)[None, ]
            end = None
            if req.end_image:
                if not END_FRAME_OK:
                    job["warning"] = "End frame ignored: the free 5B model only uses a start image (use the A14B notebook for start + end)."
                else:
                    try:
                        end_pil = load_image(req.end_image)
                    except Exception as e:
                        raise ValueError(f"Could not read the end image ({e})")
                    end = torch.from_numpy(np.array(fit_to(end_pil, w, h)).astype(np.float32) / 255.0)[None, ]
                    job["end_frame"] = True
            gc.collect()
            torch.cuda.empty_cache()
            with torch.inference_mode():
                job["stage"] = "reading the prompt"
                positive = CLIPTextEncode.encode(clip, req.prompt or "")[0]
                negative = CLIPTextEncode.encode(clip, req.negative if req.negative is not None else DEFAULT_NEGATIVE)[0]
                job["stage"] = "preparing the start image"
                if FAMILY == "a14b":
                    if end is not None:  # start + end frame: Wan fills in the motion between them
                        positive, negative, latent = nodes_wan.WanFirstLastFrameToVideo.execute(
                            positive, negative, vae, w, h, length, 1, start_image=start, end_image=end).args
                    else:
                        positive, negative, latent = nodes_wan.WanImageToVideo.execute(
                            positive, negative, vae, w, h, length, 1, start_image=start).args
                    high, low = MODELS[mode]
                    split = steps // 2  # first half: high-noise expert (motion/layout), second half: low-noise (detail)
                    job["stage"] = "animating (pass 1 of 2: motion)"
                    latent = KSamplerAdvanced.sample(high, "enable", seed, steps, cfg, P["sampler"], P["scheduler"],
                                                     positive, negative, latent, 0, split, "enable")[0]
                    job["stage"] = "animating (pass 2 of 2: details)"
                    latent = KSamplerAdvanced.sample(low, "disable", seed, steps, cfg, P["sampler"], P["scheduler"],
                                                     positive, negative, latent, split, 10000, "disable")[0]
                else:
                    latent = nodes_wan.Wan22ImageToVideoLatent.execute(vae, w, h, length, 1, start_image=start).args[0]
                    job["stage"] = "animating"
                    latent = KSampler.sample(MODELS[mode], seed, steps, cfg, P["sampler"], P["scheduler"],
                                             positive, negative, latent, 1.0)[0]
                job["stage"] = "decoding frames"
                frames = VAEDecode.decode(vae, latent)[0]
            job["stage"] = "saving the video"
            path = os.path.join(VIDEO_DIR, f"{job_id}.mp4")
            n = save_mp4(frames, fps, path)
            last_frame = frame_data_url(frames[-1])
            del frames, latent
            sound = False
            if req.sound and SOUND_OK:
                failed = add_sound(job, path, (req.sound_prompt or "").strip() or req.prompt or "", seed)
                if failed:
                    job["warning"] = f"The video is silent: {failed}."
                else:
                    sound = True
            elif req.sound:
                job["warning"] = "The video is silent: this notebook was set up without sound (ADD_SOUND in cell 1)."
            job.update(status="done", stage="done", video=f"/video/{job_id}.mp4", frames=n, sound=sound, last_frame=last_frame,
                       seconds=round(n / fps, 1), took=round(time.time() - t0))
            print(f"[{job_id}] done in {job['took']} s. {memory_report()}", flush=True)
        except Exception as e:
            traceback.print_exc()
            job.update(status="error", error=str(e))
        finally:
            current_job["id"] = None
            gc.collect()
            torch.cuda.empty_cache()


# ---------------------------------------------------------------- API for the website
from typing import Optional
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


class VideoRequest(BaseModel):
    image: str                  # start image: base64 / data URL / http(s) address
    end_image: Optional[str] = None  # optional end image (A14B): the video ends on this picture
    prompt: str = ""            # describe the motion
    negative: Optional[str] = None
    size: str = "480p"          # "480p" or "720p" (keeps the image's shape)
    seconds: float = 3.0
    mode: str = "fast"          # "fast" (4-step LoRA) or "quality" (more steps, slower)
    steps: Optional[int] = None
    cfg: Optional[float] = None
    seed: int = -1
    sound: bool = False         # add sound with MMAudio after the video is made
    sound_prompt: str = ""      # what it should sound like (empty = use the motion prompt)


@app.get("/")
def root():
    return {"status": "ok", "mode": "image_to_video", "model": CONFIG["label"]}


@app.get("/video_info")
def video_info():
    return {"mode": "image_to_video", "family": FAMILY, "label": CONFIG["label"], "gpu": CONFIG["gpu"],
            "fps": D["fps"], "max_seconds": round(D["max_frames"] / D["fps"], 1), "sizes": list(SIZE_AREAS),
            "defaults": {"size": D["size"], "seconds": D["seconds"], "mode": "fast"},
            "modes": {k: {"steps": D[k]["steps"], "cfg": D[k]["cfg"]} for k in ("fast", "quality")},
            "default_negative": DEFAULT_NEGATIVE, "supports_sound": SOUND_OK,
            "supports_end_frame": END_FRAME_OK}


@app.post("/generate_video")
def generate_video(req: VideoRequest):
    job_id = new_job()
    threading.Thread(target=run_video, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued"}


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": last_crash_message() or "Unknown job (the server restarted)"}
    return {"job_id": job_id, **{k: v for k, v in job.items() if k != "created"}}


@app.get("/video/{name}")
def get_video(name: str):
    path = os.path.join(VIDEO_DIR, os.path.basename(name))
    if not name.endswith(".mp4") or not os.path.exists(path):
        return {"error": "not found"}
    return FileResponse(path, media_type="video/mp4", filename=name)


if __name__ == "__main__":
    import uvicorn
    print("Server listening on port 8000.", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

In [ ]:
# Cell 3: start the generation server as a separate program, plus the Cloudflare tunnel.
# If the server crashes (e.g. runs out of memory) only that program dies: it's restarted automatically
# with the same URL, and this notebook keeps running. Re-running this cell restarts the server with the
# current code from cell 2. To stop everything: Runtime → Restart session.
import os, re, sys, time, subprocess, threading, urllib.request

SERVER_SCRIPT = "/content/video_server.py"
SERVER_LOG = "/content/video_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"


def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False


def log_tail(n=15):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""


class Supervisor:
    """Runs the server program and restarts it when it crashes (gives up after 4 crashes in 10 minutes)."""

    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen([sys.executable, "-u", SERVER_SCRIPT], cwd="/content/ComfyUI",
                                             stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
                code = self.proc.wait()
            if self.stopping:
                break
            why = ("ran out of memory and was stopped by the system" if code in (-9, 137)
                   else f"crashed (exit code {code})")
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = (f"The generation server {why} 4 times in 10 minutes, so it was not restarted. "
                       f"Check the log (cell 4), then run cell 3 again.")
                self.gave_up = True
            else:
                msg = (f"The generation server {why} at {time.strftime('%H:%M')} and was restarted "
                       f"(models reload in 1-3 minutes). Try a smaller size, fewer pictures or batch 1.")
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)


# (Re)start the server program
if "SUPERVISOR" in globals():
    print("Restarting the server with the current code...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading the video models in the server program (a few minutes)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("The server did not start. Last lines of its log:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see the log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

# Cloudflare tunnel (started once; the URL stays the same when the server restarts)
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()  # keep the pipe drained

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this into the website: Image to Video tab → Video server URL.")
    print("This cell is done; the server keeps running in the background.")
    print("Cell 4 shows its log and memory use. To stop it: Runtime → Restart session.")
    print("=" * 60)
else:
    print("Could not start the Cloudflare tunnel. Run this cell again.")

In [ ]:
# Cell 4 (optional): the server's recent log and memory use. Run it again to refresh.
import subprocess
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass
gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if gpu:
    print(f"GPU memory: {gpu}")
sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server program:", "running" if running else ("stopped — run cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes in the last 10 minutes: {len([t for t in sup.crashes if __import__('time').time() - t < 600])}")
print("-" * 60)
try:
    print("".join(open("/content/video_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run cell 3 first.")